# Event trigger

A batch **starts the pipeline by itself** when its delivery is complete.

```
sender writes   landing/batch_NN/_SUCCESS   (a tiny file {"files": 37}, written LAST)
      |  S3 tells EventBridge "an object was created"
      v
EventBridge rule pf-new-batch-rule   (matches only landing/batch_*/_SUCCESS in our bucket)
      |  starts, with the role pf-trigger-role
      v
Step Functions pf-batch-pipeline   input {"bucket": ..., "key": "landing/batch_NN/_SUCCESS"}
      v
Lambda checks that the batch holds the announced number of files, then the usual steps run
```
**Why a marker file?** A batch is 37 files. A trigger on "any file arrived" would start the pipeline 37 times, and it could start while the
delivery is still half done. The marker is written last and means "the delivery is complete".

Run the cells top to bottom. Releasing a batch runs the whole pipeline, which takes about 7 minutes.

## 1. Setup

In [1]:
import json
import sys
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import boto3

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# deploy.py is the same deployment code GitHub Actions uses; aws() runs the AWS CLI (see infra/aws_helpers.py)
import deploy
from aws_helpers import LAKE, REGION, SETTINGS, account_id, aws, aws_json, whoami

CONFIG = deploy.load_config()
SESSION = boto3.Session(region_name=CONFIG["region"])
RULE = SETTINGS["trigger_rule_name"]
MACHINE_ARN = f"arn:aws:states:{REGION}:{account_id()}:stateMachine:{CONFIG['state_machine']}"
ROLE_ARN = aws_json("iam", "get-role", "--role-name", "pf-trigger-role")["Role"]["Arn"]
BUILD = ROOT / "build"
BUILD.mkdir(exist_ok=True)

print("identity:", whoami())
print("rule    :", RULE)
print("bucket  :", LAKE)

identity: user/Ahmed
rule    : pf-new-batch-rule
bucket  : patientflow-datalake-zhmkbf


## 2. Deploy the Lambda that understands the marker
The updated `handler.py` accepts `{"bucket", "key"}`, reads the file count from the marker, refuses a batch whose file count differs,
and never copies the marker itself. `deploy_lambda` is the same function GitHub Actions runs.

In [2]:
deploy.deploy_lambda(SESSION, CONFIG)
function = aws_json("lambda", "get-function-configuration", "--function-name", CONFIG["lambda_name"])
assert function["State"] == "Active" and function["LastUpdateStatus"] == "Successful"
print("OK  Lambda", function["FunctionName"], "updated", function["LastModified"])

updated Lambda pf-ingest


OK  Lambda pf-ingest updated 2026-10-26T18:18:56.000+0000


## 3. Let S3 announce new files to EventBridge
S3 sends its events to EventBridge only when the bucket asks for it. This call **replaces** the bucket's whole notification
configuration, so the cell first checks that nothing else is configured.

In [3]:
current = aws_json("s3api", "get-bucket-notification-configuration", "--bucket", LAKE) or {}
others = {key: value for key, value in current.items() if key != "EventBridgeConfiguration"}
assert not others, f"the bucket already has other notifications: {others}"

aws("s3api", "put-bucket-notification-configuration", "--bucket", LAKE,
    "--notification-configuration", json.dumps({"EventBridgeConfiguration": {}}))

assert "EventBridgeConfiguration" in aws_json("s3api", "get-bucket-notification-configuration", "--bucket", LAKE)
print("OK  the bucket sends its events to EventBridge")

$ aws s3api put-bucket-notification-configuration --bucket patientflow-datalake-zhmkbf --notification-configuration {"EventBridgeConfiguration": {}}


OK  the bucket sends its events to EventBridge


## 4. The rule
An event pattern says which events the rule reacts to. Ours: S3 "Object Created" events, in our bucket, whose key looks like
`landing/batch_<anything>/_SUCCESS` (`wildcard` matching).
The **target** is the state machine. An *input transformer* reshapes the S3 event into the small JSON the Lambda expects:
`{"bucket": ..., "key": ...}`.

In [4]:
EVENT_PATTERN = {
    "source": ["aws.s3"],
    "detail-type": ["Object Created"],
    "detail": {"bucket": {"name": [LAKE]}, "object": {"key": [{"wildcard": "landing/batch_*/_SUCCESS"}]}},
}
TARGET = {
    "Id": "pipeline",
    "Arn": MACHINE_ARN,
    "RoleArn": ROLE_ARN,
    "InputTransformer": {
        "InputPathsMap": {"bucket": "$.detail.bucket.name", "key": "$.detail.object.key"},
        "InputTemplate": '{"bucket": "<bucket>", "key": "<key>"}',
    },
}

aws("events", "put-rule", "--name", RULE, "--state", "ENABLED",
    "--description", "Start the patient-flow pipeline when a batch delivery is complete (landing/batch_NN/_SUCCESS)",
    "--event-pattern", json.dumps(EVENT_PATTERN), "--query", "RuleArn", "--output", "text")
aws("events", "put-targets", "--rule", RULE, "--targets", json.dumps([TARGET]))

$ aws events put-rule --name pf-new-batch-rule --state ENABLED --description Start the patient-flow pipeline when a batch delivery is complete (landing/batch_NN/_SUCCESS) --event-pattern {"source": ["aws.s3"], "detail-type": ["Object Created"], "detail": {"bucket": {"name": ["patientflow-datalake-zhmkbf"]}, "object": {"key": [{"wildcard": "landing/batch_*/_SUCCESS"}]}}} --query RuleArn --output text


arn:aws:events:us-east-1:<ACCOUNT_ID>:rule/pf-new-batch-rule
$ aws events put-targets --rule pf-new-batch-rule --targets [{"Id": "pipeline", "Arn": "arn:aws:states:us-east-1:<ACCOUNT_ID>:stateMachine:pf-batch-pipeline", "RoleArn": "arn:aws:iam::<ACCOUNT_ID>:role/pf-trigger-role", "InputTransformer": {"InputPathsMap": {"bucket": "$.detail.bucket.name", "key": "$.detail.object.key"}, "InputTemplate": "{\"bucket\": \"<bucket>\", \"key\": \"<key>\"}"}}]


{
    "FailedEntryCount": 0,
    "FailedEntries": []
}


<aws exit code 0>

In [5]:
rule = aws_json("events", "describe-rule", "--name", RULE)
targets = aws_json("events", "list-targets-by-rule", "--rule", RULE)["Targets"]
assert rule["State"] == "ENABLED" and json.loads(rule["EventPattern"]) == EVENT_PATTERN
assert len(targets) == 1 and targets[0]["Arn"] == MACHINE_ARN and targets[0]["RoleArn"] == ROLE_ARN
print(f"OK  rule {RULE}: ENABLED, one target = the state machine, started with the role pf-trigger-role")

OK  rule pf-new-batch-rule: ENABLED, one target = the state machine, started with the role pf-trigger-role


## 5. Does the pattern match only the right events?
AWS can test a pattern against a sample event without sending anything.

In [6]:
def s3_event(key, bucket=LAKE, detail_type="Object Created"):
    return {"id": "1", "account": account_id(), "source": "aws.s3", "time": "2026-11-02T10:00:00Z", "region": REGION,
            "resources": [], "detail-type": detail_type, "detail": {"bucket": {"name": bucket}, "object": {"key": key}}}


def matches(event):
    result = aws_json("events", "test-event-pattern", "--event-pattern", json.dumps(EVENT_PATTERN), "--event", json.dumps(event))
    return result["Result"]


CASES = [
    ("the marker of batch 2", s3_event("landing/batch_02/_SUCCESS"), True),
    ("the marker of batch 10", s3_event("landing/batch_10/_SUCCESS"), True),
    ("a data file in a batch", s3_event("landing/batch_02/Abe604_Rogahn59_x.json"), False),
    ("a file called _SUCCESS in bronze", s3_event("bronze/patient_records/ingest_date=2026-11-02/_SUCCESS"), False),
    ("a file in the silver layer", s3_event("silver/patient/ingest_date=2026-11-02/part-0.parquet"), False),
    ("the marker, but another bucket", s3_event("landing/batch_02/_SUCCESS", bucket="somebody-elses-bucket"), False),
    ("the marker being deleted", s3_event("landing/batch_02/_SUCCESS", detail_type="Object Deleted"), False),
]
for label, event, expected in CASES:
    assert matches(event) is expected, label
    print(f"OK  {'starts the pipeline' if expected else 'is ignored':<20} {label}")

OK  starts the pipeline  the marker of batch 2


OK  starts the pipeline  the marker of batch 10


OK  is ignored           a data file in a batch


OK  is ignored           a file called _SUCCESS in bronze


OK  is ignored           a file in the silver layer


OK  is ignored           the marker, but another bucket


OK  is ignored           the marker being deleted


## 6. Helpers: release a batch and follow what it starts
`release_batch` does what a sender does: it counts the files of the batch and writes the marker **last**.
It then waits for the pipeline execution that EventBridge starts, and for its result.

In [7]:
def files_in_batch(batch):
    objects = aws_json("s3api", "list-objects-v2", "--bucket", LAKE, "--prefix", f"landing/batch_{batch:02d}/").get("Contents", [])
    return [o for o in objects if not o["Key"].endswith("/_SUCCESS")]


def put_marker(batch, announced_files):
    body = BUILD / "marker_body.json"
    body.write_text(json.dumps({"files": announced_files}))
    aws("s3api", "put-object", "--bucket", LAKE, "--key", f"landing/batch_{batch:02d}/_SUCCESS",
        "--body", str(body.relative_to(ROOT)), "--content-type", "application/json", "--query", "ETag", "--output", "text")


def executions_since(moment):
    found = aws_json("stepfunctions", "list-executions", "--state-machine-arn", MACHINE_ARN, "--max-results", "20")["executions"]
    return [e for e in found if datetime.fromisoformat(e["startDate"]) > moment]


def wait_for_execution(execution_arn, timeout=1500):
    deadline = time.time() + timeout
    while True:
        execution = aws_json("stepfunctions", "describe-execution", "--execution-arn", execution_arn)
        if execution["status"] != "RUNNING":
            return execution
        assert time.time() < deadline, "the execution is still running"
        time.sleep(10)


def release_batch(batch, announced_files=None):
    # Write the marker, then wait for the execution that EventBridge starts (it appears within seconds)
    moment = datetime.now(timezone.utc) - timedelta(seconds=2)
    put_marker(batch, announced_files if announced_files is not None else len(files_in_batch(batch)))
    for _ in range(30):
        started = executions_since(moment)
        if started:
            break
        time.sleep(3)
    assert started, "no execution was started within 90 seconds"
    assert len(started) == 1, f"expected one execution, found {len(started)}"
    return wait_for_execution(started[0]["executionArn"])

## 7. Release batch 2: the pipeline starts by itself
No `start-execution` here: the only action is writing the marker file.

In [8]:
count = len(files_in_batch(2))
print("batch 2 holds", count, "files; writing the marker")
run = release_batch(2)

print("status:", run["status"])
print("input the execution received:", run["input"])
print("output:", run["output"])
assert run["status"] == "SUCCEEDED", (run.get("error"), run.get("cause"))
assert json.loads(run["input"]) == {"bucket": LAKE, "key": "landing/batch_02/_SUCCESS"}
assert json.loads(run["output"]) == {"file_count": count, "batch": 2, "ingest_date": "2026-11-02"}
print("OK  writing the marker started the whole pipeline, and it succeeded")

batch 2 holds 34 files; writing the marker


$ aws s3api put-object --bucket patientflow-datalake-zhmkbf --key landing/batch_02/_SUCCESS --body build\marker_body.json --content-type application/json --query ETag --output text


"520301ed785d207e5bacf705ae903595"


status: SUCCEEDED
input the execution received: {"bucket": "patientflow-datalake-zhmkbf", "key": "landing/batch_02/_SUCCESS"}
output: {"file_count":34,"batch":2,"ingest_date":"2026-11-02"}
OK  writing the marker started the whole pipeline, and it succeeded


## 8. An incomplete delivery must fail
Batch 3's marker announces 999 files, but the batch holds 34. The Lambda must refuse it, and no later step may run.

In [9]:
run = release_batch(3, announced_files=999)
print("status:", run["status"])
print("error :", run["error"])
print("cause :", json.loads(run["cause"])["errorMessage"])
assert run["status"] == "FAILED"
assert "incomplete batch 3" in run["cause"] and "announces 999 files" in run["cause"]

history = aws_json("stepfunctions", "get-execution-history", "--execution-arn", run["executionArn"])["events"]
entered = [e["stateEnteredEventDetails"]["name"] for e in history if e["type"].endswith("StateEntered")]
print("steps entered:", entered)
assert "BronzeToSilver" not in entered
print("OK  the pipeline stopped at the first step; Glue and the merges did not run")

$ aws s3api put-object --bucket patientflow-datalake-zhmkbf --key landing/batch_03/_SUCCESS --body build\marker_body.json --content-type application/json --query ETag --output text


"70f72583e5c077c008770d0ee0fc80b1"


status: FAILED
error : RuntimeError
cause : incomplete batch 3: the marker announces 999 files but s3://patientflow-datalake-zhmkbf/landing/batch_03/ holds 34


steps entered: ['Ingest', 'PipelineFailed']
OK  the pipeline stopped at the first step; Glue and the merges did not run


## 9. Only creating the marker starts a run
Remove the wrong marker of batch 3. A deletion is a different kind of event, so nothing must start.

In [10]:
moment = datetime.now(timezone.utc) - timedelta(seconds=2)
aws("s3api", "delete-object", "--bucket", LAKE, "--key", "landing/batch_03/_SUCCESS")
time.sleep(45)
assert executions_since(moment) == []
print("OK  deleting the marker started nothing")

remaining = [o["Key"] for o in aws_json("s3api", "list-objects-v2", "--bucket", LAKE, "--prefix", "landing/").get("Contents", [])
             if o["Key"].endswith("/_SUCCESS")]
print("markers in landing/:", remaining)

$ aws s3api delete-object --bucket patientflow-datalake-zhmkbf --key landing/batch_03/_SUCCESS


OK  deleting the marker started nothing


markers in landing/: ['landing/batch_02/_SUCCESS']


## 10. Recent executions

In [11]:
for e in aws_json("stepfunctions", "list-executions", "--state-machine-arn", MACHINE_ARN, "--max-results", "5")["executions"]:
    print(f"{e['startDate'][11:19]}  {e['status']:<10}  {e['name'][:40]}")
print("\nStep Functions console > pf-batch-pipeline > Executions: executions started by EventBridge have a long random name.")
print("EventBridge console > Rules > " + RULE + " shows the rule and its target.")

23:27:02  FAILED      7d56832a-d1f9-57e1-5a4a-442e2e30c1f7_cb5
23:19:31  SUCCEEDED   17e5a7cd-c037-2659-762d-ddaad7446126_42e
22:26:49  SUCCEEDED   batch-4-1793035607
22:26:47  SUCCEEDED   batch-3-1793035605
22:26:42  FAILED      batch-5-1793035600

Step Functions console > pf-batch-pipeline > Executions: executions started by EventBridge have a long random name.
EventBridge console > Rules > pf-new-batch-rule shows the rule and its target.


## How a real delivery works

| Step | Who |
|---|---|
| 1. Upload the batch's data files to `landing/batch_NN/` | the sender (any number of files, in any order) |
| 2. Write `landing/batch_NN/_SUCCESS` containing `{"files": <how many were uploaded>}` | the sender, **last** |
| 3. Everything after that | automatic |

To run a batch again, write its marker again (a new "created" event). A batch is processed once per marker.
Release one batch at a time when you can: two at once work (the second waits at Glue), but they write the same Iceberg tables.

**Switching it off:** disable the rule (`aws events disable-rule --name pf-new-batch-rule`); markers then start nothing.